# Data Cleaning

The dataset will be transformed into a clean and analysis-ready format. Each of the sections will have mentions of the process.

**Author**: Nikolas Antoniou<br>
**Project**: Reatil Shop Analytics

_______

## Setup & Imports

In [6]:
# necessary libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
# importing premade audit pipeline
sys.path.append("../notebooks/premade_models")
from code_audit import run_audit

In [10]:
# figures in Times New Roman
mpl.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman"],
    "axes.titlesize"    : 14,
    "axes.labelsize"    : 12,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "legend.fontsize"   : 10,
    "figure.titlesize"  : 16,
})

sns.set_theme(style="whitegrid", palette="muted", font="Times New Roman")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

RAW_PATH   = "../data/inventoryANDsales2025.csv"
CLEAN_PATH = "../data/clean/clothing_clean.csv"

__________

## Baseline Audit

All transformations are applied to a working copy `df`.
The audit is re-run as a baseline to confirm we are starting from the same state documented in `00_DataAudit.ipynb`.

In [11]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()

In [12]:
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

Loaded: 3,500 rows × 12 columns



In [13]:
print(run_audit(RAW_PATH))

CSV DATA AUDIT REPORT
File : /Users/nicolasdilegardia/Desktop/EDA-and-Classification/Retail-Insights Clothing Shop/data/inventoryANDsales2025.csv
Date : 2026-06-12 17:20:07

────────────────────────────────────────────────────────────
  1. Shape
────────────────────────────────────────────────────────────
    Rows: 3,500   Columns: 12

────────────────────────────────────────────────────────────
  2. Data Types
────────────────────────────────────────────────────────────
    Id                                  object
    name                                object
    season                              object
    price                               float64
    product_category                    object
    size                                object
    quantity                            int64
    quantities_sold                     int64
    collection_family                   object
    sex                                 object
    color                               object
    disc

________

## Baseline Cleaning

Two flags in `00_DataAudit.ipynb` are not real data problems:

- **`sex` unexpected values**: the audit's valid set used lowercase while the data uses title case (`Men`, `Women`). No fix needed.
- **`quantities_sold > quantity`**: `quantity` reflects *current remaining stock*, not original units received. Sales exceeding current stock is expected mid-season. No fix needed.

Fix 1 — Drop True Duplicates

The audit identified **1,564 rows** sharing the same `name` and `size`.
These are not multi-size variants of the same product — they are repeated entries
for the exact same SKU. Keeping them would inflate every groupby aggregation
(sales totals, size distributions, color counts).

**Decision:** drop all but the first occurrence, keyed on `(name, size)`.
**Assertion:** zero `(name, size)` duplicates must remain after the fix.

In [17]:
df = df_raw.copy()

before = len(df)
df = df.drop_duplicates(subset=["name", "size"], keep="first").reset_index(drop=True)
after  = len(df)
remaining = df.duplicated(subset=["name", "size"]).sum()

print(f"Rows before : {before:,}")
print(f"Rows after  : {after:,}  (dropped {before - after:,})")
print(f"Remaining (name+size) duplicates: {remaining}")
assert remaining == 0, "Duplicate check failed — investigate before continuing."
print("Assertion passed.")

Rows before : 3,500
Rows after  : 1,936  (dropped 1,564)
Remaining (name+size) duplicates: 0
Assertion passed.


Fix 2 — Separate Size Systems

The `size` column contains two incompatible sizing conventions:
- **EU numeric** (36–46): footwear and some structured garments
- **Alpha** (XS, S, M, L, XL, XXL, One Size): apparel

Mixing them in a single column makes H1 (size demand by category) meaningless —
you cannot rank "M" against "42". A new column `size_type` separates the two systems
so each can be analysed independently within its product category.

**Decision:** add `size_type` = `"numeric"` or `"alpha"`. The original `size` column is kept untouched.  
**Assertion:** every row must be assigned exactly one type — no nulls, no unknowns.

In [18]:
numeric_mask = df["size"].str.match(r"^\d+$")

df["size_type"] = np.where(numeric_mask, "numeric", "alpha")

# breakdown
counts = df.groupby(["size_type", "size"]).size().rename("rows")
print(df["size_type"].value_counts().to_string())
print()
print(counts.to_string())

# assertion
assert df["size_type"].isnull().sum() == 0, "Nulls found in size_type."
assert set(df["size_type"].unique()) == {"numeric", "alpha"}, "Unexpected size_type value."
print("\nAssertions passed.")

size_type
alpha      1752
numeric     184

size_type  size    
alpha      L           260
           M           315
           One Size     76
           S           288
           XL          266
           XS          272
           XXL         275
numeric    36           14
           37           14
           38           11
           39           14
           40           23
           41           24
           42           25
           43           21
           44           13
           45           15
           46           10

Assertions passed.


Fix 3 — Quantity Outlier Investigation

The audit flagged `quantity` with **57 IQR outliers and 13 Z-score outliers** (skew 1.006).
Z-score > 3 represents the most extreme values and warrants manual inspection.

Two possible explanations:
- **Bulk SKUs**: accessories or basics stocked in high volume — real data, keep as-is
- **Data entry error**: a digit added by mistake (e.g. 5600 instead of 560) — must cap or correct

In [19]:
from scipy import stats as st

In [21]:
q_zscore = np.abs(st.zscore(df["quantity"]))

In [22]:
outliers = df[q_zscore > 3].sort_values("quantity", ascending=False)

print(f"Z-score > 3 outliers in quantity: {len(outliers)}\n")
display(outliers[["name", "product_category", "size", "size_type",
                   "quantity", "quantities_sold", "price", "discount"]].reset_index(drop=True))

Z-score > 3 outliers in quantity: 7



,name,product_category,size,size_type,quantity,quantities_sold,price,discount
0,Patch Graphic Sweatshirt,Sweatshirt,XXL,alpha,561,32,151.9900,0
1,Tribe Print T-Shirt,T-Shirt,XS,alpha,555,31,44.9900,0
2,Morgex Quilted Pants,Pants,S,alpha,542,56,102.9900,0
3,Circular Pants,Pants,L,alpha,537,39,150.9900,0
4,Rainforest Stripe Shorts,Shorts,M,alpha,535,47,58.9900,0
5,Box Logo Half-Zip Shoes,Shoes,38,numeric,529,55,151.9900,0
6,Box Logo Half-Zip Shoes,Shoes,41,numeric,529,70,188.9900,0


In [23]:
print(f"\nQuantity stats (full dataset):")
print(df["quantity"].describe().round(1).to_string())
print(f"\nOutlier quantity range: {outliers['quantity'].min()} – {outliers['quantity'].max()}")


Quantity stats (full dataset):
count   1936.0000
mean     152.8000
std      123.3000
min        2.0000
25%       53.0000
50%      117.0000
75%      224.0000
max      561.0000

Outlier quantity range: 529 – 561


We have to really understand if these results reflect outliers or high demand products which implies high quantities and sales. Many shops work through carryovers and high quantities of famous products.

In [24]:
# verifying the legitimacy of outliers
df["sell_through"] = (df["quantities_sold"] / (df["quantity"] + df["quantities_sold"])).round(3)

In [25]:
outliers = df[q_zscore > 3].sort_values("quantity", ascending=False)

In [26]:
print("─ High-quantity SKUs (Z > 3) ─")
display(outliers[["name", "product_category", "size_type",
                   "quantity", "quantities_sold", "sell_through"]].reset_index(drop=True))
print(f"\nMean sell-through — outliers    : {outliers['sell_through'].mean():.1%}")
print(f"Mean sell-through — rest of data: {df[q_zscore <= 3]['sell_through'].mean():.1%}")
print("\n✓ No cleaning required. High quantities reflect genuine bulk inventory.")

─ High-quantity SKUs (Z > 3) ─


,name,product_category,size_type,quantity,quantities_sold,sell_through
0,Patch Graphic Sweatshirt,Sweatshirt,alpha,561,32,0.0540
1,Tribe Print T-Shirt,T-Shirt,alpha,555,31,0.0530
2,Morgex Quilted Pants,Pants,alpha,542,56,0.0940
3,Circular Pants,Pants,alpha,537,39,0.0680
4,Rainforest Stripe Shorts,Shorts,alpha,535,47,0.0810
5,Box Logo Half-Zip Shoes,Shoes,numeric,529,55,0.0940
6,Box Logo Half-Zip Shoes,Shoes,numeric,529,70,0.1170



Mean sell-through — outliers    : 8.0%
Mean sell-through — rest of data: 52.7%

✓ No cleaning required. High quantities reflect genuine bulk inventory.


From the results it is clear that the supposed **high-demands** are underperforming because of the 8% VS the 52.7%, which implies that the shop over-ordered them and now these products consume the total quantity. 

This results leads to some meanings:<br>
- These are not popular bulk SKUs — they are potential dead stock
- The shop committed a lot of capital to items that are not moving
- This is worth surfacing in the EDA and business recommendation

In [27]:
print(f"\nMean sell-through — outliers    : {outliers['sell_through'].mean():.1%}")
print(f"Mean sell-through — rest of data: {df[q_zscore <= 3]['sell_through'].mean():.1%}")
print("\nHigh-quantity SKUs show significantly lower sell-through (8% vs 53%).")
print("  These are likely over-ordered items — potential dead stock.")
print("  Rows are kept as valid data. Flag for business recommendation in EDA.")


Mean sell-through — outliers    : 8.0%
Mean sell-through — rest of data: 52.7%

High-quantity SKUs show significantly lower sell-through (8% vs 53%).
  These are likely over-ordered items — potential dead stock.
  Rows are kept as valid data. Flag for business recommendation in EDA.


_____

## Final Verification

Before any exports, the consistency of the clean dataset must be confirmed:<br>
- Row count reflects the 1,564 duplicates removed
- No nulls introduced by any transformation
- New columns (`size_type`, `sell_through`) are complete and correctly typed
- Shape and dtypes are as expected

In [28]:
# shape
print("SHAPE\n")
print(f"  Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")

SHAPE

  Rows: 1,936   Columns: 14


In [29]:
# nulls
nulls = df.isnull().sum()
if nulls.sum() == 0:
    print("No nulls exist in any column.")
else:
    print(nulls[nulls > 0])

No nulls exist in any column.


In [30]:
# defining the sell_through column
print(df[["size_type", "sell_through"]].dtypes.to_string())
print(f"\n  size_type values : {df['size_type'].unique()}")
print(f"  sell_through range: {df['sell_through'].min()} – {df['sell_through'].max()}")

size_type        object
sell_through    float64

  size_type values : ['alpha' 'numeric']
  sell_through range: 0.045 – 0.97


In [31]:
print(df.dtypes.to_string())

assert df.isnull().sum().sum() == 0, "Nulls found — do not export."
assert df.shape[0] == 1936, f"Unexpected row count: {df.shape[0]}"
assert "size_type" in df.columns
assert "sell_through" in df.columns

Id                    object
name                  object
season                object
price                float64
product_category      object
size                  object
quantity               int64
quantities_sold        int64
collection_family     object
sex                   object
color                 object
discount               int64
size_type             object
sell_through         float64


______

## Exporting

The clean file is written to `data/clean/` and never overwrites the raw source.
All downstream notebooks (`02_EDA`, `03_Classification`, `04_FinalModel`) load from this file.

In [32]:
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)

df.to_csv(CLEAN_PATH, index=False)

verify = pd.read_csv(CLEAN_PATH)
print(f"Exported : {CLEAN_PATH}")
print(f"Shape    : {verify.shape[0]:,} rows × {verify.shape[1]} columns")
print(f"Columns  : {list(verify.columns)}")
assert verify.shape == df.shape, "Export shape mismatch."
print("\nExport verified.")

Exported : ../data/clean/clothing_clean.csv
Shape    : 1,936 rows × 14 columns
Columns  : ['Id', 'name', 'season', 'price', 'product_category', 'size', 'quantity', 'quantities_sold', 'collection_family', 'sex', 'color', 'discount', 'size_type', 'sell_through']

Export verified.
